In [1]:
import pandas as pd
import requests
import yfinance as yf
import numpy as np
import re
from collections import Counter

In [2]:
headers = {"User-Agent": "Mozilla/5.0 (compatible; my-script/1.0)"}
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
html = requests.get(url, headers=headers).text
table = pd.read_html(html)[0]
table['Symbol'] = table['Symbol'].str.replace('.', '-').tolist()
table

/var/folders/5l/vsm1wvjn4qdg25qwjfcj83kr0000gn/T/ipykernel_99718/2584062866.py:4: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  table = pd.read_html(html)[0]


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927


In [3]:
table.to_csv('../data/tickers.csv')

### Tickers
**Semiconductors**: NVDA (NVIDIA), AMD (Advanced Micro Devices), INTC (Intel), ASML (ASML Holding), TSM (Taiwan Semiconductor Manufacturing), AVGO (Broadcom), 
    MRVL (Marvell Technology), ARM (Arm Holdings), SMCI (Super Micro Computer)<br>
**Hyperscalars**: MSFT (Microsoft), GOOGL (Google), AMZN (Amazon), META (Facebook) <br>
**AI Pure Plays**: PLTR (Palantir), AI (C3.ai), SOUN (Soundhound AI) <br>
**Infrastructure**: DELL (Dell Technologies), ANET (Arista Networks), HPE (Hewlett Packard Enterprise)

In [4]:
tickers = pd.read_csv('../data/tickers.csv')
tickers = tickers.drop(columns=['Unnamed: 0'])

ai_tickers = {
    'semis':   ['NVDA','AMD','INTC','ASML','TSM','AVGO','MRVL','ARM','SMCI'], # semiconductors
    'hypers':  ['MSFT','GOOGL','AMZN','META'], # hyperscalar
    'pure':    ['PLTR','AI','SOUN'], # AI pure plays
    'infra':   ['DELL','ANET','HPE'] # infrastructure
}
all_tickers = [t for group in ai_tickers.values() for t in group]
all_tickers

['NVDA',
 'AMD',
 'INTC',
 'ASML',
 'TSM',
 'AVGO',
 'MRVL',
 'ARM',
 'SMCI',
 'MSFT',
 'GOOGL',
 'AMZN',
 'META',
 'PLTR',
 'AI',
 'SOUN',
 'DELL',
 'ANET',
 'HPE']

In [5]:
# includes years before GenAI Revolution
prices = yf.download(
    tickers=all_tickers + ['SOXX','QQQ'],
    start='2018-01-01',
    end='2026-01-01',
    auto_adjust=True,
    progress=False
)
prices.to_csv('../data/prices_raw.csv')
prices.shape  # (trading_days, n_tickers)


(2011, 105)

In [6]:
close  = prices['Close']
volume = prices['Volume']
soxx   = close['SOXX']
qqq    = close['QQQ']

### Clean Data

In [7]:
# Remove tickers with more than 5% NaN values
clean_prices = close.loc[:, close.isna().mean() <= 0.05]

# Forward fill short gaps of 1-2 days
clean_prices = clean_prices.ffill(limit=2)
clean_prices = clean_prices.dropna()
clean_prices
clean_prices.to_csv('../data/clean_prices.csv')


In [8]:
clean_prices

Ticker,AMD,AMZN,ANET,ASML,AVGO,DELL,GOOGL,HPE,INTC,META,MRVL,MSFT,NVDA,QQQ,SMCI,SOXX,TSM
Date,,,,,,,,,,,,,,,,,
2018-01-02,10.980000,59.450500,14.439375,163.685852,21.086660,21.049421,53.188862,11.300009,39.330379,179.840714,21.104073,78.699905,4.922529,150.222183,2.145000,53.624989,33.674377
2018-01-03,11.550000,60.209999,14.725000,164.929245,21.317265,21.271584,54.096321,11.369618,37.995583,183.062424,21.760010,79.066177,5.246500,151.681839,2.150000,54.562698,34.240814
2018-01-04,12.120000,60.479500,14.543125,166.467224,21.324373,21.649519,54.306450,11.648058,37.298801,182.725388,22.254335,79.762054,5.274155,151.947189,2.165000,54.870148,34.060200
2018-01-05,11.880000,61.457001,14.798125,168.419693,21.450729,21.807844,55.026569,11.640325,37.559048,185.223480,21.864582,80.750954,5.318849,153.473206,2.170000,55.183735,34.856506
2018-01-08,12.280000,62.343498,15.691250,169.303864,21.502064,21.802734,55.220844,11.462431,37.559048,186.641022,22.026184,80.833351,5.481822,154.070389,2.180000,55.617245,34.840088
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-24,215.039993,232.380005,130.770004,1061.797241,349.486877,127.426529,313.681671,24.212580,36.160000,666.361328,86.384567,485.856323,188.380234,623.144287,30.549999,305.806305,297.264252
2025-12-26,214.990005,232.520004,131.839996,1069.001953,351.392883,128.280151,313.102478,24.262114,36.200001,662.108948,86.234749,485.547699,190.297897,623.104309,30.639999,305.706390,301.283478
2025-12-29,215.610001,232.070007,134.149994,1062.275635,348.658630,126.513359,313.152405,24.103603,36.680000,657.517151,85.655464,484.940430,187.990707,620.088135,30.080000,304.987183,299.373383


### Log Returns

In [9]:
log_ret = np.log(clean_prices / clean_prices.shift(1)).dropna()
log_ret

Ticker,AMD,AMZN,ANET,ASML,AVGO,DELL,GOOGL,HPE,INTC,META,MRVL,MSFT,NVDA,QQQ,SMCI,SOXX,TSM
Date,,,,,,,,,,,,,,,,,
2018-01-03,0.050610,0.012694,0.019588,0.007568,0.010877,0.010499,0.016917,0.006141,-0.034527,0.017756,0.030608,0.004643,0.063739,0.009670,0.002328,0.017335,0.016681
2018-01-04,0.048172,0.004466,-0.012428,0.009282,0.000333,0.017611,0.003877,0.024195,-0.018509,-0.001843,0.022463,0.008763,0.005257,0.001748,0.006952,0.005619,-0.005289
2018-01-05,-0.020001,0.016033,0.017382,0.011661,0.005908,0.007286,0.013173,-0.000664,0.006953,0.013579,-0.017669,0.012322,0.008438,0.009993,0.002307,0.005699,0.023110
2018-01-08,0.033116,0.014322,0.058603,0.005236,0.002390,-0.000234,0.003524,-0.015401,0.000000,0.007624,0.007364,0.001020,0.030181,0.003884,0.004598,0.007825,-0.000471
2018-01-09,-0.038179,0.004665,-0.004311,-0.005400,-0.013943,-0.012967,-0.001275,0.002022,-0.025352,-0.002180,-0.000863,-0.000679,-0.000270,0.000062,-0.025553,-0.009888,-0.006145
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-24,0.000651,0.001033,-0.004197,0.003460,0.002573,0.005938,-0.000827,-0.005712,-0.005241,0.003917,-0.013665,0.002400,-0.003176,0.002921,-0.006850,0.003862,0.006211
2025-12-26,-0.000232,0.000602,0.008149,0.006762,0.005439,0.006677,-0.001848,0.002044,0.001106,-0.006402,-0.001736,-0.000635,0.010128,-0.000064,0.002942,-0.000327,0.013430
2025-12-29,0.002880,-0.001937,0.017369,-0.006312,-0.007812,-0.013869,0.000159,-0.006555,0.013173,-0.006959,-0.006740,-0.001251,-0.012198,-0.004852,-0.018446,-0.002355,-0.006360


In [10]:
# SOXX: holds 30 largest semiconductor companies
# QQQ: holds 100 largest non-financial companies on the NASDAQ, heavily weighted towards big tech
def make_features(close: pd.Series,
                     volume: pd.Series,
                     soxx: pd.Series,
                     qqq: pd.Series) -> pd.DataFrame:
    df = pd.DataFrame(index=close.index)

    # Lag returns: percent change over x days
    for lag in [1, 3, 5, 10, 20]:
        df[f'ret_{lag}d'] = close.pct_change(lag)

    # Rolling volatility
    df['vol_5d']  = close.pct_change().rolling(5).std()
    df['vol_20d'] = close.pct_change().rolling(20).std()

    # RSI (14-day): Relative Strength Index; measures magnitude and recent price changes in stocks
    delta = close.pct_change()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    df['rsi_14'] = 100 - (100 / (1 + gain / loss))

    # How does the stock compare to SOXX and QQQ?
    df['rel_soxx'] = close.pct_change(20) - soxx.pct_change(20)
    df['rel_qqq']  = close.pct_change(20) - qqq.pct_change(20)

    # Volume spike: number of shares traded on a given day
    df['vol_spike'] = volume / volume.rolling(20).mean()

    # 52-week high proximity: the highest price traded at over the last 252 trading days (1 year)
    df['pct_from_52wk_high'] = close / close.rolling(252).max() - 1

    return df.dropna()

In [11]:
all_feature_dfs = []
for ticker in all_tickers:
    if ticker not in close.columns:
        continue
    ticker_feats = make_features(
        close  = close[ticker],
        volume = volume[ticker],
        soxx   = soxx,
        qqq    = qqq
    )
    ticker_feats['Ticker'] = ticker      
    all_feature_dfs.append(ticker_feats)

In [12]:
features_df = pd.concat(all_feature_dfs).reset_index()
features_df = features_df.set_index(['Date','Ticker'])

In [13]:
# Did this stock go up over the next 5 days?
features_df['Target'] = (
    features_df.groupby('Ticker')['ret_1d']
    .transform(lambda x: x.shift(-1).rolling(5).sum() > 0)
).astype(int)

In [14]:
features_df.to_csv('../data/features_df.csv')

In [15]:
# VALIDATE FEATURES_DF
# Quick sanity check — does the date index behave as expected?
dates = features_df.index.get_level_values('Date')

print(f"Total rows:   {len(features_df)}")
print(f"Date range:   {dates.min().date()} → {dates.max().date()}")
print(f"Trading days: {dates.nunique()}")
print(f"Tickers:      {features_df.index.get_level_values('Ticker').nunique()}")
print(f"Avg rows/ticker: {len(features_df) / features_df.index.get_level_values('Ticker').nunique():.0f}")

# Check target balance — should be roughly 50/50
target_dist = features_df['Target'].value_counts(normalize=True)
print(f"\nTarget distribution:\n{target_dist}")
# Expect: 0 → ~47%, 1 → ~53% (markets go up slightly more than down)

# Check for nulls
null_pct = features_df.isnull().mean() * 100
print(f"\nNull % per feature:\n{null_pct.round(2)}")
# All should be 0.0% after dropna() in make_ai_features

# Check no future leakage — target should be NaN for the last row per ticker
print(f"Any NaN in last row targets: {features_df.groupby(level='Ticker').tail(1)['Target'].isna().any()}")
# Expected output: Any NaN in last row targets: False

# The definitive null check is this one instead:
print(f"Total NaN in entire target column: {features_df['Target'].isna().sum()}")
# Expected output: Total NaN in entire target column: 0

Total rows:   29487
Date range:   2019-01-02 → 2025-12-31
Trading days: 1760
Tickers:      19
Avg rows/ticker: 1552

Target distribution:
Target
1    0.563435
0    0.436565
Name: proportion, dtype: float64

Null % per feature:
ret_1d                0.0
ret_3d                0.0
ret_5d                0.0
ret_10d               0.0
ret_20d               0.0
vol_5d                0.0
vol_20d               0.0
rsi_14                0.0
rel_soxx              0.0
rel_qqq               0.0
vol_spike             0.0
pct_from_52wk_high    0.0
Target                0.0
dtype: float64
Any NaN in last row targets: False
Total NaN in entire target column: 0


In [16]:
# Should print 0
print(features_df['Target'].isna().sum())

# Should only contain 0 and 1
print(features_df['Target'].unique())

# Should be roughly 53% ones
print(features_df['Target'].mean().round(3))

0
[0 1]
0.563


### Walk Forward Validation Framework

In [17]:
def walk_forward_splits(df, n_splits=5, test_months=3):
    """
    Expanding window walk-forward splits.
    Each fold: train on all data up to cutoff, test on next 3 months.
    NEVER shuffle — time order must be preserved.
    """
    all_dates = df.index.get_level_values('date').unique()
    all_dates = pd.DatetimeIndex(all_dates).sort_values()

    total = len(all_dates)
    fold_size = total // (n_splits + 1)

    splits = []
    for i in range(n_splits):
        train_end = fold_size * (i + 1)
        test_end  = fold_size * (i + 2)
        train_idx = all_dates[:train_end]
        test_idx  = all_dates[train_end:test_end]

        train = df[df.index.get_level_values('date').isin(train_idx)]
        test  = df[df.index.get_level_values('date').isin(test_idx)]
        splits.append((train, test))
        print(f"Fold {i+1}: train {train_idx[0].date()}→{train_idx[-1].date()}, "
              f"test {test_idx[0].date()}→{test_idx[-1].date()}")
    return splits

### Continuous News Sentiment with FinnHub

In [18]:
# imports
from transformers import pipeline
from datetime import datetime, timedelta
import finnhub
from dotenv import load_dotenv
import os

In [19]:
load_dotenv()

True

In [20]:
client = finnhub.Client(api_key=os.getenv("FINNHUB_API_KEY"))

In [21]:
# date, headline, and summary of most recent 50 news per ticker
def get_ai_news(ticker, start_date, end_date):
    start_ts = int(datetime.strptime(start_date, "%Y-%m-%d").timestamp())
    end_ts   = int(datetime.strptime(end_date,   "%Y-%m-%d").timestamp())

    news = client.company_news(ticker, _from=start_date, to=end_date)

    articles = []
    for item in news[:50]:
        articles.append({
            'date':     datetime.fromtimestamp(item['datetime']).date(),
            'headline': item['headline'],
            'summary':  item.get('summary', '')
        })
    return pd.DataFrame(articles)

In [22]:
finbert = pipeline('sentiment-analysis', model='ProsusAI/finbert', truncation=True, max_length=512)

Device set to use mps:0


In [23]:
def score_headlines(df):
    texts = (df['headline'] + '. ' + df['summary']).tolist()
    results = finbert(texts)
    df['sentiment'] = [r['label'].lower() for r in results]
    df['score'] = [r['score'] if r['label'] == 'positive' else -r['score'] if r['label'] == 'negative' else 0 for r in results]
    daily = df.groupby('date').agg(
        news_sentiment = ('score', 'mean'),
        news_count = ('score', 'count'),
        pct_positive = ('sentiment', lambda x: (x=='positive').mean())
    )
    return daily

In [24]:
def get_sentiment_for_ticker(ticker, df_ticker):
    # find the beginning and end dates for each ticker
    start_date = df_ticker.index.get_level_values('Date').min().strftime("%Y-%m-%d")
    end_date   = df_ticker.index.get_level_values('Date').max().strftime("%Y-%m-%d")
    
    news_df = get_ai_news(ticker, start_date, end_date)
    if news_df.empty:
        return pd.DataFrame()
    
    # provides sentiment rating based on 50 most recent news for each ticker
    daily_sentiment = score_headlines(news_df)
    daily_sentiment['Ticker'] = ticker
    return daily_sentiment


In [25]:
# Loop over all tickers
all_sentiment = []
for ticker in features_df.index.get_level_values('Ticker').unique():
    df_ticker = features_df.xs(ticker, level='Ticker')
    sentiment = get_sentiment_for_ticker(ticker, df_ticker)
    if not sentiment.empty:
        all_sentiment.append(sentiment)

sentiment_df = pd.concat(all_sentiment)
sentiment_df.index = pd.to_datetime(sentiment_df.index)
sentiment_df
sentiment_df.to_csv('../data/sentiment_df.csv')

In [26]:
sentiment_df

,news_sentiment,news_count,pct_positive,Ticker
date,,,,
2025-12-30,0.335194,14,0.571429,NVDA
2025-12-31,0.107661,36,0.388889,NVDA
2025-12-29,-0.041530,22,0.318182,AMD
2025-12-30,0.139328,19,0.368421,AMD
2025-12-31,0.308277,9,0.555556,AMD
...,...,...,...,...
2025-12-26,0.953101,1,1.000000,HPE
2025-12-28,-0.768126,1,0.000000,HPE
2025-12-29,0.931547,1,1.000000,HPE


In [27]:
features_df

,,ret_1d,ret_3d,ret_5d,ret_10d,ret_20d,vol_5d,vol_20d,rsi_14,rel_soxx,rel_qqq,vol_spike,pct_from_52wk_high,Target
Date,Ticker,,,,,,,,,,,,,
2019-01-02,NVDA,0.020374,0.038500,0.071923,-0.051261,-0.166493,0.023554,0.034328,37.207847,-0.107088,-0.083557,0.785101,-0.528766,0
2019-01-03,NVDA,-0.060417,-0.042350,-0.038393,-0.128964,-0.247295,0.033031,0.034169,30.211061,-0.110360,-0.119161,1.104590,-0.557236,0
2019-01-04,NVDA,0.064068,0.020150,0.038271,-0.016750,-0.133155,0.045219,0.035031,40.655563,-0.080480,-0.078587,0.933377,-0.528870,0
2019-01-07,NVDA,0.052941,0.052709,0.072951,0.061436,-0.094068,0.049557,0.037384,48.847598,-0.063841,-0.044049,1.128762,-0.503928,0
2019-01-08,NVDA,-0.024896,0.092507,0.047415,0.079185,-0.052707,0.052560,0.034712,48.297105,-0.055655,-0.044009,1.240790,-0.516278,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-24,HPE,-0.005696,0.000409,0.023605,-0.026643,0.151076,0.014461,0.017580,65.514413,0.069940,0.125055,0.190724,-0.063353,1
2025-12-26,HPE,0.002046,-0.008502,0.029987,0.003964,0.144324,0.013753,0.017617,62.905712,0.092509,0.127354,0.418928,-0.061437,0
2025-12-29,HPE,-0.006533,-0.010171,-0.004093,0.025401,0.119174,0.007448,0.017731,56.790174,0.088752,0.115266,0.511510,-0.067568,0


In [28]:
# merge sentiment df back to features df
df = features_df.reset_index()
df['Date'] = pd.to_datetime(df['Date'])
df = df.merge(
    sentiment_df.reset_index().rename(columns={'date': 'Date'}),
    on=['Date', 'Ticker'],
    how='left'
)
df = df.set_index(['Date', 'Ticker'])

# Fill missing sentiment (days with no news)
df[['news_sentiment', 'new_count', 'pct_positive']] = df[['news_sentiment', 'news_count', 'pct_positive']].fillna(0)
df

,,ret_1d,ret_3d,ret_5d,ret_10d,ret_20d,vol_5d,vol_20d,rsi_14,rel_soxx,rel_qqq,vol_spike,pct_from_52wk_high,Target,news_sentiment,news_count,pct_positive,new_count
Date,Ticker,,,,,,,,,,,,,,,,,
2019-01-02,NVDA,0.020374,0.038500,0.071923,-0.051261,-0.166493,0.023554,0.034328,37.207847,-0.107088,-0.083557,0.785101,-0.528766,0,0.000000,NaN,0.0,0.0
2019-01-03,NVDA,-0.060417,-0.042350,-0.038393,-0.128964,-0.247295,0.033031,0.034169,30.211061,-0.110360,-0.119161,1.104590,-0.557236,0,0.000000,NaN,0.0,0.0
2019-01-04,NVDA,0.064068,0.020150,0.038271,-0.016750,-0.133155,0.045219,0.035031,40.655563,-0.080480,-0.078587,0.933377,-0.528870,0,0.000000,NaN,0.0,0.0
2019-01-07,NVDA,0.052941,0.052709,0.072951,0.061436,-0.094068,0.049557,0.037384,48.847598,-0.063841,-0.044049,1.128762,-0.503928,0,0.000000,NaN,0.0,0.0
2019-01-08,NVDA,-0.024896,0.092507,0.047415,0.079185,-0.052707,0.052560,0.034712,48.297105,-0.055655,-0.044009,1.240790,-0.516278,1,0.000000,NaN,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-24,HPE,-0.005696,0.000409,0.023605,-0.026643,0.151076,0.014461,0.017580,65.514413,0.069940,0.125055,0.190724,-0.063353,1,0.941576,3.0,1.0,3.0
2025-12-26,HPE,0.002046,-0.008502,0.029987,0.003964,0.144324,0.013753,0.017617,62.905712,0.092509,0.127354,0.418928,-0.061437,0,0.953101,1.0,1.0,1.0
2025-12-29,HPE,-0.006533,-0.010171,-0.004093,0.025401,0.119174,0.007448,0.017731,56.790174,0.088752,0.115266,0.511510,-0.067568,0,0.931547,1.0,1.0,1.0


In [29]:
# Domain keyword dictionaries
GPU_DEMAND_KEYWORDS = [
    # current & next-gen hardware
    'h100', 'h200', 'h20',
    'blackwell', 'b100', 'b200', 'gb200', 'gb300',
    'rubin', 'vera rubin', 'nvl72',
    'dgx', 'hgx',
    # demand signals
    'gpu demand', 'gpu supply', 'gpu allocation',
    'data center revenue', 'data center demand',
    'inference demand', 'inference workload',
    'training compute', 'compute demand',
    # software/platform
    'cuda', 'tensorrt', 'triton', 'nim microservices',
    'hopper architecture', 'blackwell architecture'
    # other GPU demand words
    'cloud ai', 'ai infrastructure', 'ai accelerator', 'tensor processing', 'custom ai chip'
]

CAPEX_UP_KEYWORDS = [
    'increasing capex', 'raised capex', 'accelerating capex',
    'accelerating investment', 'capital expenditure increase',
    'expanding capacity', 'record investment',
    'investing heavily', 'multiyear investment',
    'committed to investing', 'significant investment in ai',
    'capital investment', 'infrastructure investment', 'ai investment'
]

CAPEX_DOWN_KEYWORDS = [
    'reducing capex', 'cutting investment', 'slowing spend',
    'deferring investment', 'pausing investment',
    'capital efficiency', 'disciplined spending',
    'revisiting capex', 'moderating investment'
]

COMPETITIVE_RISK_KEYWORDS = [
    # rival chips by name
    'amd instinct', 'gaudi', 'trainium', 'inferentia', 'maia',
    'google tpu', 'axion',
    # strategic alternatives
    'custom silicon', 'in-house chip', 'in-house ai chip',
    'merchant silicon', 'proprietary chip',
    'competitive alternative', 'alternative to nvidia'
]

In [30]:
def extract_domain_signals(transcript_text):
    text_lower = transcript_text.lower()
    signals = {}

    # Raw mention counts (normalize by transcript length)
    word_count = len(text_lower.split())
    signals['gpu_mentions']    = sum(text_lower.count(k) for k in GPU_DEMAND_KEYWORDS) / word_count * 1000
    signals['capex_up_score']  = sum(text_lower.count(k) for k in CAPEX_UP_KEYWORDS) / word_count * 1000
    signals['capex_down_score']= sum(text_lower.count(k) for k in CAPEX_DOWN_KEYWORDS) / word_count * 1000
    competitor_keys = [k for k in COMPETITIVE_RISK_KEYWORDS if k != ticker.lower()]
    signals['competitor_mentions'] = sum(text_lower.count(k) for k in competitor_keys) / word_count * 1000
    signals['capex_net']       = signals['capex_up_score'] - signals['capex_down_score'] 

    # Paragraph-level: find sentences mentioning AI demand
    ai_sentences = [s for s in transcript_text.split('.')
                    if any(k in s.lower() for k in ['artificial intelligence','ai demand','ai revenue'])]
    signals['ai_sentence_ratio'] = len(ai_sentences) / max(len(transcript_text.split('.')),1)

    return signals

In [31]:
# imports
from edgar import Company, set_identity
from bs4 import BeautifulSoup

In [32]:
set_identity('Heidi heidit1688@gmail.com')

In [33]:
# # get the transcript for the extract domain signals function
# def get_earnings_transcripts(ticker):
#     """
#     Pull only 8-K filings containing Item 2.02 (Results of Operations)
#     which are the earnings releases most likely to contain transcript-like text.
#     """
#     company = Company(ticker)
#     filings = company.get_filings(form="8-K")
    
#     results = []
#     for filing in filings:
#         try:
#             obj = filing.obj()
            
#             # Filter: only keep filings with Item 2.02 (only earnings related)
#             items = getattr(obj, 'items', [])
#             if '2.02' not in str(items):
#                 continue
            
#             text = filing.text()
            
#             # Secondary filter: must contain at least one domain keyword
#             # so we don't waste signal slots on boilerplate 8-Ks
#             text_lower = text.lower()
#             has_signal = any(k in text_lower for k in 
#                            GPU_DEMAND_KEYWORDS + CAPEX_UP_KEYWORDS + 
#                            CAPEX_DOWN_KEYWORDS + COMPETITOR_KEYWORDS)
#             if not has_signal:
#                 continue
                
#             results.append({
#                 'ticker': ticker,
#                 'date': filing.filing_date,
#                 'text': text
#             })
            
#         except Exception as e:
#             print(f"Skipped {ticker} {filing.filing_date}: {e}")
#             continue
    
#     print(f"{ticker}: {len(results)} qualifying filings")
#     return results

In [34]:
# for filing in filings[:20]:
#     print(filing.filing_date, "| items:", filing.items)

In [35]:
# earnings_filing = filings[1]  # 2026-05-20
# print(earnings_filing.attachments)

In [36]:
# # test text extraction on this filing
# for attachment in earnings_filing.attachments:
#     if 'EX-99' in attachment.document_type:
#         print(attachment.document_type, "|", attachment.document)
#         text = attachment.text()
#         print(text[:1000])
#         print("total length:", len(text))
#         break

In [37]:
# cleaned get earnings transcript 
def get_earnings_transcripts(ticker):
    company = Company(ticker)
    filings = company.get_filings(form="8-K")
    
    results = []
    for filing in filings:
        try:
            # only earnings releases
            if '2.02' not in str(filing.items):
                continue
            
            for attachment in filing.attachments:
                if 'EX-99' not in attachment.document_type:
                    continue
                
                text = attachment.text()
                if not text or len(text) < 200:
                    continue
                
                results.append({
                    'Ticker': ticker,
                    'Date': pd.to_datetime(filing.filing_date),
                    'Text': text
                })
                break  # one exhibit per filing
                
        except Exception as e:
            print(f"Skipped {ticker} {filing.filing_date}: {e}")
            continue
    
    print(f"{ticker}: {len(results)} qualifying filings")
    return results

In [38]:
all_transcript_signals = []

for ticker in all_tickers:
    transcripts = get_earnings_transcripts(ticker)
    for t in transcripts:
        signals = extract_domain_signals(t['Text'])
        signals['Ticker'] = t['Ticker']
        signals['Date'] = t['Date']
        all_transcript_signals.append(signals)

transcript_signals_df = (pd.DataFrame(all_transcript_signals).set_index(['Date', 'Ticker']).sort_index())

transcript_signals_df.to_csv('../data/transcript_signals_df.csv')
transcript_signals_df

NVDA: 93 qualifying filings


AMD: 94 qualifying filings


INTC: 108 qualifying filings
ASML: 0 qualifying filings
TSM: 0 qualifying filings


AVGO: 33 qualifying filings


MRVL: 23 qualifying filings
ARM: 0 qualifying filings


...

SOUN: 18 qualifying filings


DELL: 41 qualifying filings


ANET: 48 qualifying filings


HPE: 53 qualifying filings


gpu_mentions  capex_up_score  capex_down_score  \
Date       Ticker                                                   
2004-10-07 AMD         0.000000             0.0               0.0   
2004-10-12 INTC        0.000000             0.0               0.0   
2004-10-21 AMZN        0.000000             0.0               0.0   
           MSFT        0.000000             0.0               0.0   
2004-10-27 NVDA        0.000000             0.0               0.0   
...                         ...             ...               ...   
2026-05-27 MRVL        0.000000             0.0               0.0   
2026-05-28 DELL        0.000000             0.0               0.0   
2026-06-01 HPE         0.000000             0.0               0.0   
2026-06-03 AI          0.000000             0.0               0.0   
           AVGO        0.266596             0.0               0.0   

                   competitor_mentions  capex_net  ai_sentence_ratio  
Date       Ticker                                                     
2004-10-07 AMD                     0.0        0.0           0.000000  
2004-10-12 INTC                    0.0        0.0           0.000000  
2004-10-21 AMZN                    0.0        0.0           0.000000  
           MSFT                    0.0        0.0           0.000000  
2004-10-27 NVDA                    0.0        0.0           0.000000  
...                                ...        ...                ...  
2026-05-27 MRVL                    0.0        0.0           0.002227  
2026-05-28 DELL                    0.0        0.0           0.000000  
2026-06-01 HPE                     0.0        0.0           0.004739  
2026-06-03 AI                      0.0        0.0           0.007299  
           AVGO                    0.0        0.0           0.000000  

[935 rows x 6 columns]

In [39]:
df

,,ret_1d,ret_3d,ret_5d,ret_10d,ret_20d,vol_5d,vol_20d,rsi_14,rel_soxx,rel_qqq,vol_spike,pct_from_52wk_high,Target,news_sentiment,news_count,pct_positive,new_count
Date,Ticker,,,,,,,,,,,,,,,,,
2019-01-02,NVDA,0.020374,0.038500,0.071923,-0.051261,-0.166493,0.023554,0.034328,37.207847,-0.107088,-0.083557,0.785101,-0.528766,0,0.000000,NaN,0.0,0.0
2019-01-03,NVDA,-0.060417,-0.042350,-0.038393,-0.128964,-0.247295,0.033031,0.034169,30.211061,-0.110360,-0.119161,1.104590,-0.557236,0,0.000000,NaN,0.0,0.0
2019-01-04,NVDA,0.064068,0.020150,0.038271,-0.016750,-0.133155,0.045219,0.035031,40.655563,-0.080480,-0.078587,0.933377,-0.528870,0,0.000000,NaN,0.0,0.0
2019-01-07,NVDA,0.052941,0.052709,0.072951,0.061436,-0.094068,0.049557,0.037384,48.847598,-0.063841,-0.044049,1.128762,-0.503928,0,0.000000,NaN,0.0,0.0
2019-01-08,NVDA,-0.024896,0.092507,0.047415,0.079185,-0.052707,0.052560,0.034712,48.297105,-0.055655,-0.044009,1.240790,-0.516278,1,0.000000,NaN,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-24,HPE,-0.005696,0.000409,0.023605,-0.026643,0.151076,0.014461,0.017580,65.514413,0.069940,0.125055,0.190724,-0.063353,1,0.941576,3.0,1.0,3.0
2025-12-26,HPE,0.002046,-0.008502,0.029987,0.003964,0.144324,0.013753,0.017617,62.905712,0.092509,0.127354,0.418928,-0.061437,0,0.953101,1.0,1.0,1.0
2025-12-29,HPE,-0.006533,-0.010171,-0.004093,0.025401,0.119174,0.007448,0.017731,56.790174,0.088752,0.115266,0.511510,-0.067568,0,0.931547,1.0,1.0,1.0


In [40]:
# merge back to original dataframe
df = df.reset_index()

df = pd.merge_asof(
    df.sort_values('Date'),
    transcript_signals_df.reset_index().rename(columns={'date': 'Date'}).sort_values('Date'),
    on='Date',
    by='Ticker',
    direction='backward'
)

with_domain_keywords = df.set_index(['Date', 'Ticker'])
with_domain_keywords[['gpu_mentions','capex_up_score','capex_down_score', 'capex_net','competitor_mentions','ai_sentence_ratio']] = \
    with_domain_keywords[['gpu_mentions','capex_up_score','capex_down_score', 'capex_net','competitor_mentions','ai_sentence_ratio']].fillna(0)

In [41]:
with_domain_keywords.columns

Index(['ret_1d', 'ret_3d', 'ret_5d', 'ret_10d', 'ret_20d', 'vol_5d', 'vol_20d',
       'rsi_14', 'rel_soxx', 'rel_qqq', 'vol_spike', 'pct_from_52wk_high',
       'Target', 'news_sentiment', 'news_count', 'pct_positive', 'new_count',
       'gpu_mentions', 'capex_up_score', 'capex_down_score',
       'competitor_mentions', 'capex_net', 'ai_sentence_ratio'],
      dtype='object')

In [42]:
with_domain_keywords.to_csv('../data/ai_sector_features_df.csv')

In [43]:
signal_cols = ['gpu_mentions', 'capex_up_score', 'capex_down_score', 
               'capex_net', 'competitor_mentions', 'ai_sentence_ratio']

# how many rows have non-zero values
print((transcript_signals_df[signal_cols] != 0).sum())
print()
# summary stats
print(transcript_signals_df[signal_cols].describe())

gpu_mentions           111
capex_up_score          87
capex_down_score         6
capex_net               88
competitor_mentions     55
ai_sentence_ratio      203
dtype: int64

       gpu_mentions  capex_up_score  capex_down_score   capex_net  \
count    935.000000      935.000000        935.000000  935.000000   
mean       0.176224        0.041275          0.001374    0.039901   
std        0.909909        0.152019          0.018247    0.152578   
min        0.000000        0.000000          0.000000   -0.230681   
25%        0.000000        0.000000          0.000000    0.000000   
50%        0.000000        0.000000          0.000000    0.000000   
75%        0.000000        0.000000          0.000000    0.000000   
max       12.601512        1.331558          0.367579    1.331558   

       competitor_mentions  ai_sentence_ratio  
count           935.000000         935.000000  
mean              0.045569           0.001843  
std               0.278431           0.005584  
min       

In [44]:
# see signal totals per ticker
transcript_signals_df.groupby('Ticker')[signal_cols].sum().round(2)

,gpu_mentions,capex_up_score,capex_down_score,capex_net,competitor_mentions,ai_sentence_ratio
Ticker,,,,,,
AI,0.91,0.00,0.23,-0.23,0.00,0.08
AMD,5.73,0.22,0.00,0.22,23.27,0.07
AMZN,1.75,0.12,0.00,0.12,8.03,0.10
ANET,0.00,0.00,0.00,0.00,2.82,0.10
AVGO,1.07,0.00,0.00,0.00,0.00,0.07
DELL,0.51,0.22,0.22,-0.01,0.23,0.02
GOOGL,1.94,1.40,0.00,1.40,0.00,0.04
HPE,0.12,0.00,0.11,-0.11,0.00,0.19
INTC,2.65,1.39,0.72,0.68,2.51,0.04


In [45]:
missing = set(all_tickers) - set(transcript_signals_df.index.get_level_values('Ticker').unique())
print(missing)

{'ARM', 'ASML', 'TSM'}


In [46]:
# should show non-zero values persisting across ~63 trading days per quarter
with_domain_keywords[with_domain_keywords.index.get_level_values('Ticker') == 'NVDA']['gpu_mentions'].value_counts().head(10)

gpu_mentions
0.816327    68
1.183432    68
3.423592    67
3.681304    65
2.151116    65
1.802343    64
1.111729    64
0.554170    64
4.790419    64
3.102962    64
Name: count, dtype: int64